# 02 — Train and evaluate YOLOv8s on facade materials

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarEAbdelaal/M4U3-CV-Facade-Materials/blob/main/notebooks/02_training_eval.ipynb)

Detects 6 facade materials: `brick`, `cladding_panel`, `exposed_concrete`, `glass`, `painted_render`, `stone_cladding`.

**Reproducible with no credentials.** The dataset is downloaded from the project's GitHub Release and checked against its SHA256. No Roboflow account, API key or Colab Secret is needed.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

| Cell | What it does |
|---|---|
| 1 | Install the pinned Ultralytics version |
| 2 | Settings (model, epochs, image size, new-image URLs) |
| 3 | Download the frozen dataset from the GitHub Release and verify SHA256 |
| 4 | Clean labels: convert SAM polygons to boxes, drop boxes under 4 px |
| 5 | Record the environment (versions, GPU, time) |
| 6 | Train YOLOv8s |
| 7 | Evaluate: precision / recall / mAP50 / mAP50-95, overall and per class |
| 8 | Save training curves and confusion matrix |
| 9 | Predict on 10 validation images |
| 10 | Predict on new images that are not in the dataset |
| 11 | Package results and weights for download |

Outputs go to `/content/results/`, matching the repo's `results/` folder.

In [ ]:
# Cell 1 — install (version pinned for reproducibility)
%pip install -q ultralytics==8.4.163

In [ ]:
# Cell 2 — settings
from pathlib import Path

MODEL = "yolov8s.pt"      # pretrained on COCO; fine-tuned here (transfer learning)
EPOCHS = 50
IMGSZ = 640
BATCH = 16
SEED = 0
FRACTION = 1.0            # share of training images to use (1.0 = all)

# If no GPU is available: set QUICK_RUN = True for a 5-epoch verification run,
# and put the Release URL of the real best.pt in WEIGHTS_URL so evaluation and
# inference use the fully trained model (allowed by the brief; say so in the README).
QUICK_RUN = False
WEIGHTS_URL = ""
if QUICK_RUN:
    EPOCHS = 5

MIN_BOX_PX = 4            # boxes narrower or shorter than this (at 640 px) are annotation noise
CONF = 0.25               # confidence threshold for the prediction images

# 5 images that are NOT in the dataset (e.g. Wikimedia Commons file URLs), for the new-image evidence
NEW_IMAGE_URLS = [
    # "https://upload.wikimedia.org/...jpg",
]

RESULTS = Path("/content/results")
for sub in ["curves", "evidence/val_predictions", "evidence/new_predictions", "weights"]:
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)
print(f"{MODEL}, epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, quick_run={QUICK_RUN}")

## Dataset (primary, keyless path)

This is the course's standard download cell. Only `DATA_URL` and `SHA256` are project-specific.

- Release: `v1.0`, asset `facade-materials-v3-yolov8.zip` (contents: Roboflow project `facade-materials-3mrrt`, **version 4**)
- The checksum proves the file is the exact one the model was trained on.

In [ ]:
# Cell 3 — download the frozen dataset from the GitHub Release and verify it
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/OmarEAbdelaal/M4U3-CV-Facade-Materials/releases/download/v1.0/facade-materials-v3-yolov8.zip"
SHA256 = "134e41be8c310452bbe8ea33fd55a891c6e29ca863f3c9b8df3827f59bb951cd"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

## Label cleanup

Two problems were found when the released dataset was inspected (see `docs/project_log.md`):

1. **Mixed label formats.** Some objects were annotated with SAM masks. Roboflow's YOLOv8 export kept them as polygons, and 181 label files mix polygons with plain boxes. The YOLOv8 detection loader then reads the box lines as polygons too, which corrupts them. Fix: convert every polygon to its tightest box.
2. **Annotation fragments.** About 3,500 boxes are under 4 px wide or tall at 640 px, mostly SAM fragments on `glass` and `stone_cladding`. They are removed as noise.

The cleanup runs on the extracted copy only; the released zip (and its checksum) is unchanged. It is safe to run more than once.

In [ ]:
# Cell 4 — convert polygons to boxes and drop sub-4-px boxes
import collections

def clean_labels(root, min_px=MIN_BOX_PX, img_size=IMGSZ):
    min_norm = min_px / img_size
    stats = collections.Counter()
    for lbl in Path(root).rglob("labels/*.txt"):
        out = []
        for line in lbl.read_text().splitlines():
            p = line.split()
            if not p:
                continue
            if len(p) > 5:                                  # polygon -> tightest box
                xs = [float(v) for v in p[1::2]]
                ys = [float(v) for v in p[2::2]]
                x1, x2 = max(0.0, min(xs)), min(1.0, max(xs))
                y1, y2 = max(0.0, min(ys)), min(1.0, max(ys))
                p = [p[0], f"{(x1 + x2) / 2:.6f}", f"{(y1 + y2) / 2:.6f}", f"{x2 - x1:.6f}", f"{y2 - y1:.6f}"]
                stats["polygons_converted"] += 1
            if len(p) != 5:
                stats["malformed_dropped"] += 1
                continue
            if min(float(p[3]), float(p[4])) < min_norm:  # annotation fragment
                stats["tiny_dropped"] += 1
                continue
            out.append(" ".join(p))
            stats["boxes_kept"] += 1
        lbl.write_text("\n".join(out) + ("\n" if out else ""))
    return stats

print(dict(clean_labels(dataset_root)))

# boxes and images per class, per split, after cleanup
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][k] for k in sorted(cfg["names"])]
import pandas as pd
rows = []
for split in ["train", "valid", "test"]:
    lbls = list((dataset_root / split / "labels").glob("*.txt"))
    if not lbls:
        continue
    boxes, imgs = collections.Counter(), collections.Counter()
    for l in lbls:
        seen = set()
        for line in l.read_text().splitlines():
            if line.strip():
                c = names[int(line.split()[0])]
                boxes[c] += 1
                seen.add(c)
        for c in seen:
            imgs[c] += 1
    for c in names:
        rows.append({"split": split, "class": c, "images": imgs[c], "boxes": boxes[c]})
    print(f"{split}: {len(lbls)} images")
counts = pd.DataFrame(rows).pivot(index="class", columns="split", values=["images", "boxes"])
counts.to_csv(RESULTS / "dataset_counts.csv")
counts

In [ ]:
# Cell 5 — record the environment (for the README "Reproducibility proof")
import json, platform, datetime, torch, ultralytics

run_info = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "python": platform.python_version(),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "seed": SEED,
    "quick_run": QUICK_RUN, "dataset_url": DATA_URL, "dataset_sha256": SHA256,
}
(RESULTS / "run_info.json").write_text(json.dumps(run_info, indent=2))
print(json.dumps(run_info, indent=2))
if not torch.cuda.is_available():
    print("\nNo GPU found. Runtime -> Change runtime type -> T4 GPU, or set QUICK_RUN = True in cell 2.")

In [ ]:
# Cell 6 — train
import time
from ultralytics import YOLO

t0 = time.time()
model = YOLO(MODEL)
model.train(
    data=str(cfg_path), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
    deterministic=True, fraction=FRACTION, plots=True,
    project="/content/runs", name="yolov8s_facade", exist_ok=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
train_minutes = (time.time() - t0) / 60
print(f"Training took {train_minutes:.1f} min. Best weights: {BEST}")

if QUICK_RUN and WEIGHTS_URL:
    BEST = Path("/content/best_published.pt")
    urllib.request.urlretrieve(WEIGHTS_URL, BEST)
    print("QUICK_RUN: evaluation and inference use the published weights:", WEIGHTS_URL)

run_info["train_minutes"] = round(train_minutes, 1)
(RESULTS / "run_info.json").write_text(json.dumps(run_info, indent=2))

In [ ]:
# Cell 7 — evaluate: overall and per-class precision / recall / mAP
best = YOLO(str(BEST))
rows = []
for split in ["val"] + (["test"] if "test" in cfg else []):
    m = best.val(data=str(cfg_path), split=split, imgsz=IMGSZ, batch=BATCH, plots=True,
                 project="/content/runs", name=f"eval_{split}", exist_ok=True, verbose=False)
    rows.append({"split": split, "class": "all", "precision": m.box.mp, "recall": m.box.mr,
                 "mAP50": m.box.map50, "mAP50-95": m.box.map})
    for i, c in enumerate(m.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        rows.append({"split": split, "class": names[int(c)], "precision": p, "recall": r,
                     "mAP50": ap50, "mAP50-95": ap})
metrics = pd.DataFrame(rows).round(3)
metrics.to_csv(RESULTS / "metrics.csv", index=False)
metrics

In [ ]:
# Cell 8 — save training curves and confusion matrices
import shutil
from IPython.display import Image as IPImage, display

for f in list(RUN_DIR.glob("*.png")) + list(RUN_DIR.glob("*.csv")):
    shutil.copy(f, RESULTS / "curves" / f.name)
for split in ["val", "test"]:
    ev = Path("/content/runs") / f"eval_{split}"
    for f in ev.glob("confusion_matrix*.png"):
        shutil.copy(f, RESULTS / "curves" / f"{split}_{f.name}")
print(sorted(p.name for p in (RESULTS / "curves").iterdir()))

for name in ["results.png", "confusion_matrix_normalized.png"]:
    if (RESULTS / "curves" / name).exists():
        display(IPImage(filename=str(RESULTS / "curves" / name), width=900))

In [ ]:
# Cell 9 — predictions on 10 validation images (same 10 every run)
import random
from PIL import Image

val_images = sorted((dataset_root / "valid" / "images").glob("*"))
random.Random(SEED).shuffle(val_images)
for img in val_images[:10]:
    r = best.predict(str(img), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    Image.fromarray(r.plot()[:, :, ::-1]).save(RESULTS / "evidence" / "val_predictions" / f"{img.stem}_pred.jpg")
print(len(list((RESULTS / "evidence" / "val_predictions").glob("*.jpg"))), "validation predictions saved")

In [ ]:
# Cell 10 — predictions on new images that are not in the dataset
import requests

new_dir = Path("/content/new_images")
new_dir.mkdir(exist_ok=True)
headers = {"User-Agent": "M4U3-facade-materials/1.0 (student project)"}
for i, url in enumerate(NEW_IMAGE_URLS):
    dst = new_dir / f"new_{i:02d}.jpg"
    if not dst.exists():
        resp = requests.get(url, headers=headers, timeout=60)
        resp.raise_for_status()
        dst.write_bytes(resp.content)

new_images = sorted(new_dir.glob("*.jpg"))
if not new_images:
    print("No new images: add 5 URLs to NEW_IMAGE_URLS in cell 2 (or upload files to /content/new_images).")
for img in new_images:
    r = best.predict(str(img), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    Image.fromarray(r.plot()[:, :, ::-1]).save(RESULTS / "evidence" / "new_predictions" / f"{img.stem}_pred.jpg")
print(len(list((RESULTS / "evidence" / "new_predictions").glob("*.jpg"))), "new-image predictions saved")

In [ ]:
# Cell 11 — package results and weights
shutil.copy(BEST, RESULTS / "weights" / "best.pt")
shutil.make_archive("/content/results_package", "zip", RESULTS)
print("results -> /content/results_package.zip")
print("weights -> /content/results/weights/best.pt  (attach to a GitHub Release; do not commit it)")
try:
    from google.colab import files
    files.download("/content/results_package.zip")
except Exception:
    pass